In [0]:
%sql
CREATE OR REPLACE VIEW datalakehouse.gold.vw_ipca AS
WITH base AS (
    SELECT
        DATA_REFERENCIA AS data,
        VALOR AS ipca_mensal,
        YEAR(DATA_REFERENCIA) AS ano,
        MONTH(DATA_REFERENCIA) AS mes
    FROM datalakehouse.silver.tbl_ipca_serie_433
),
var_mensal AS (
    SELECT
        *,
        -- Evita divisão por zero usando try_divide()
        try_divide(
            ipca_mensal,
            LAG(ipca_mensal) OVER (ORDER BY data)
        ) - 1 AS ipca_var_mensal
    FROM base
),
acumulados AS (
    SELECT
        *,
        -- acumulado 12 meses usando produto das variações
        EXP(SUM(LN(1 + ipca_var_mensal))
            OVER (ORDER BY data ROWS BETWEEN 11 PRECEDING AND CURRENT ROW)
        ) - 1 AS ipca_acum_12m,

        SUM(ipca_var_mensal)
            OVER (PARTITION BY ano ORDER BY data
                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS ipca_acum_ano
    FROM var_mensal
),
media_moveis AS (
    SELECT
        *,
        AVG(ipca_mensal) OVER (ORDER BY data ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS ipca_media_movel_3m,
        AVG(ipca_mensal) OVER (ORDER BY data ROWS BETWEEN 11 PRECEDING AND CURRENT ROW) AS ipca_media_movel_12m
    FROM acumulados
)
SELECT * FROM media_moveis;
